# 04 — Baseline Models

**Phase 3 / Step 5 of the Homophily-Aware GNN for Fraud Ring Detection project.**

Non-graph baselines on the temporal splits — the bar the GNN (Step 6) has
to beat. Models: Logistic Regression, Random Forest, XGBoost, MLP, and a
stratified Dummy classifier as the reference floor.

**Setup (all constraints honoured)**
- **Splits are temporal** (train steps 1–34, val 35–41, test 42–49) —
  never random.
- **Supervised on labelled nodes only** (illicit vs licit). Unknown nodes
  (class 3) stay in the dataset — they are never deleted — but have no
  target and take no part in training/evaluation.
- **Features:** all 183 (Time step + 93 local + 72 aggregate + 17 named
  stats).
- **Missing values:** median imputation fit on TRAIN only + one binary
  missingness-indicator column for the 17 named-stat block. The label is
  never used for imputation.
- **Standardisation:** StandardScaler fit on train only; applied to
  LogReg/MLP only (tree models use unscaled features).
- **No SMOTE, no oversampling, no resampling** — class weighting only.
- **Model selection on VAL using PR-AUC;** final numbers on TEST.
- **Primary metric PR-AUC** (average precision). Accuracy is never
  headlined — only 2.23% of transactions are illicit.

**Outputs**
- Figures → `../results/figures/baselines/` · reports →
  `../results/baseline_results.json`, `../results/baseline_summary.md`.

In [1]:
import sys
sys.path.insert(0, "../scripts")
import run_baselines as rb

print("scripts/run_baselines.py loaded")
print("named-stat columns:", len(rb.NAMED_STATS))

scripts/run_baselines.py loaded
named-stat columns: 17


In [2]:
# 1. Load data (read-only): all 183 features + labels + temporal splits
df = rb.load_all()
feat_cols = rb.feature_columns(df)
print(f"rows: {len(df):,}  feature columns: {len(feat_cols)}")
print("splits:", df["split"].value_counts().to_dict())
print("classes:", df["class"].value_counts().to_dict())
print("missing cells in named stats:",
      int(df[rb.NAMED_STATS].isna().sum().sum()))
print("rows missing all 17 named stats:",
      int(df[rb.NAMED_STATS].isna().all(axis=1).sum()))

rows: 203,769  feature columns: 183
splits: {'train': 136265, 'test': 36824, 'val': 30680}
classes: {3: 157205, 2: 42019, 1: 4545}
missing cells in named stats: 16405
rows missing all 17 named stats: 965


In [3]:
# 2. Labelled train/val/test matrices (unknowns excluded — no target)
Xy = rb.split_xy(df, feat_cols)
for name in ("train", "val", "test"):
    X, y = Xy[name]
    print(f"{name:5s} labelled: {len(y):,}  "
          f"(illicit {int(y.sum()):,}, licit {int((y == 0).sum()):,})  "
          f"X shape {X.shape}")
print("\nunknown nodes stay in df (never deleted) but have no target: "
      f"{int((df['class'] == 3).sum()):,} rows")

train labelled: 29,894  (illicit 3,462, licit 26,432)  X shape (29894, 183)
val   labelled: 7,829  (illicit 675, licit 7,154)  X shape (7829, 183)
test  labelled: 8,841  (illicit 408, licit 8,433)  X shape (8841, 183)

unknown nodes stay in df (never deleted) but have no target: 157,205 rows


In [4]:
# 3. Train all models (selection on val PR-AUC) — takes a few minutes
results, fitted = rb.train_all(Xy, feat_cols)
best = max(results, key=lambda n: results[n]["val"]["pr_auc"])
print(f"\nbest model by val PR-AUC: {best} "
      f"({results[best]['val']['pr_auc']:.4f})")

train labelled: 29,894 (illicit 3,462, licit 26,432); scale_pos_weight=7.635
[logreg           ] fit   13.5s  val PR-AUC=0.3823  test PR-AUC=0.2025
[random_forest    ] fit   34.5s  val PR-AUC=0.9277  test PR-AUC=0.5407
[xgboost          ] fit   14.0s  val PR-AUC=0.9344  test PR-AUC=0.5652
[mlp              ] fit   14.9s  val PR-AUC=0.6404  test PR-AUC=0.3948
[dummy_stratified ] fit    0.0s  val PR-AUC=0.0882  test PR-AUC=0.0461

best model by val PR-AUC: xgboost (0.9344)


In [5]:
# 4. Results table — val and test, PR-AUC primary
print(f"{'model':17s} {'val PR-AUC':>10s} {'test PR-AUC':>11s} "
      f"{'test ROC-AUC':>12s} {'test F1@0.5':>11s} "
      f"{'prec@0.5':>9s} {'rec@0.5':>8s}")
for name in sorted(results, key=lambda n: results[n]["val"]["pr_auc"],
                   reverse=True):
    v, t = results[name]["val"], results[name]["test"]
    print(f"{name:17s} {v['pr_auc']:10.4f} {t['pr_auc']:11.4f} "
          f"{t['roc_auc']:12.4f} {t['f1_at_0.5']:11.4f} "
          f"{t['precision_at_0.5']:9.4f} {t['recall_at_0.5']:8.4f}")
print("\n(accuracy is deliberately not reported — 2.23% illicit makes it "
      "meaningless here)")

model             val PR-AUC test PR-AUC test ROC-AUC test F1@0.5  prec@0.5  rec@0.5
xgboost               0.9344      0.5652       0.8542      0.5928    0.7615   0.4853
random_forest         0.9277      0.5407       0.8346      0.5860    0.9609   0.4216
mlp                   0.6404      0.3948       0.8541      0.3432    0.6940   0.2279
logreg                0.3823      0.2025       0.8571      0.2144    0.1238   0.7990
dummy_stratified      0.0882      0.0461       0.4991      0.0644    0.0455   0.1103

(accuracy is deliberately not reported — 2.23% illicit makes it meaningless here)


In [6]:
# 5. Figures (saved to ../results/figures/baselines/)
fig_paths = rb.make_figures(results, feat_cols, fitted)
for fp in fig_paths:
    print("saved", fp.split("/")[-1])

saved baselines_pr_curves.png
saved baselines_roc_curves.png
saved baselines_importance_random_forest.png
saved baselines_importance_xgboost.png
saved baselines_confusion_matrices.png


In [7]:
# 6. Reports (JSON + Markdown summary)
rb.write_reports(results, feat_cols, fig_paths, best)
print("wrote results/baseline_results.json, results/baseline_summary.md")
print(f"best model: {best} — val PR-AUC "
      f"{results[best]['val']['pr_auc']:.4f}, test PR-AUC "
      f"{results[best]['test']['pr_auc']:.4f}")

wrote results/baseline_results.json, results/baseline_summary.md
best model: xgboost — val PR-AUC 0.9344, test PR-AUC 0.5652


## Summary

- **Best baseline: XGBoost** — val PR-AUC 0.9344, **test PR-AUC 0.5652**,
  test ROC-AUC 0.8542, F1@0.5 0.5928. Random Forest is second
  (val 0.9277, test 0.5407). MLP (0.6404 / 0.3948) and LogReg
  (0.3823 / 0.2025) trail far behind; Dummy sits at chance (0.0882 / 0.0461).
- **Temporal gap is real:** every model drops sharply from val (steps 35–41)
  to test (steps 42–49) — e.g. XGBoost 0.934 → 0.565. Future transactions
  are genuinely harder; this is exactly what the temporal split is for.
- **ROC-AUC flatters:** LogReg has the highest test ROC-AUC (0.8571) but
  the second-worst PR-AUC (0.2025) — on 2.23% positives, PR-AUC is the
  honest metric.
- **Artifacts:** `results/baseline_results.json`,
  `results/baseline_summary.md`, figures in `results/figures/baselines/`
- **Next:** `05_gnn_experiments.ipynb` — GCN / GAT / GraphSAGE and
  homophily-aware variants (Step 6, not started). The GNN must beat
  XGBoost's test PR-AUC of 0.5652.
